# Day 3 — Window Functions and Analytical SQL

**Target:** 3–4 years experience SQL hands-on practice.

**Database:** MySQL 8.x. The tables/data are created in a separate setup SQL file; this notebook focuses on syllabus, commands, explanations, practice queries, assignments, and solutions.


## Learning method

1. Run each query manually in MySQL Workbench/CLI.
2. Before executing, predict the result.
3. Change one condition and rerun.
4. Check execution plans where applicable.
5. Complete assignments without looking at the solution first.


## Important execution commands

```sql
SHOW DATABASES;
USE company_sql_practice;
SHOW TABLES;
SELECT DATABASE();
DESCRIBE employees;
```

The setup file contains the five-table schema and large sample dataset.


## Syllabus

- Why window functions exist
- OVER() and PARTITION BY
- ORDER BY inside OVER
- ROW_NUMBER
- RANK
- DENSE_RANK
- NTILE
- LAG and LEAD
- FIRST_VALUE and LAST_VALUE
- Running totals
- Moving averages
- Percentage of department/company total
- Top-N per group
- Deduplication with ROW_NUMBER
- Month-over-month and year-over-year patterns
- Window frames: ROWS BETWEEN
- Combining GROUP BY with window functions


## Practice Queries

Each query is intentionally written as a workplace-style example. Read the explanation before changing it.


### Practice 1

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Ranking employees inside each department
SELECT employee_id, employee_name, department_id, salary,
       RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS salary_rank,
       DENSE_RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS dense_salary_rank,
       ROW_NUMBER() OVER (PARTITION BY department_id ORDER BY salary DESC, employee_id) AS row_num
FROM employees;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 2

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Top 3 earners per department
WITH ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (
               PARTITION BY department_id
               ORDER BY salary DESC, employee_id
           ) AS rn
    FROM employees e
)
SELECT *
FROM ranked
WHERE rn <= 3;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 3

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Running payroll by hire date
SELECT employee_id, employee_name, hire_date, salary,
       SUM(salary) OVER (
           ORDER BY hire_date, employee_id
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_payroll
FROM employees;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 4

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Salary compared with department average
SELECT employee_id, employee_name, department_id, salary,
       ROUND(AVG(salary) OVER (PARTITION BY department_id), 2) AS dept_avg,
       ROUND(salary - AVG(salary) OVER (PARTITION BY department_id), 2) AS diff_from_dept_avg
FROM employees;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 5

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- LAG: salary/hire-date style comparison
SELECT employee_id, employee_name, department_id, salary,
       LAG(salary) OVER (
           PARTITION BY department_id
           ORDER BY salary
       ) AS previous_salary
FROM employees;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 6

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Revenue share by customer
WITH customer_revenue AS (
    SELECT c.customer_id, c.customer_name,
           SUM(oi.quantity * oi.unit_price) AS revenue
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY c.customer_id, c.customer_name
)
SELECT customer_name, revenue,
       ROUND(100 * revenue / SUM(revenue) OVER (), 2) AS pct_of_total
FROM customer_revenue
ORDER BY revenue DESC;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


## Assignments

1. Return top 3 salaries per department using ROW_NUMBER.
2. Return top 3 salaries per department while preserving ties using DENSE_RANK.
3. Calculate each employee's salary percentile using NTILE(4).
4. Calculate month-over-month revenue change using LAG.
5. Find the first and latest order for every customer.
6. Calculate a 3-order moving average of customer order value.
7. Remove duplicate customer records using ROW_NUMBER logic.


## Solutions

Try the assignment first. Then compare with the reference pattern below. Adapt names/filters to your dataset.


### Solution 1


In [ ]:
WITH x AS (
 SELECT e.*, ROW_NUMBER() OVER(PARTITION BY department_id ORDER BY salary DESC, employee_id) rn
 FROM employees e
) SELECT * FROM x WHERE rn<=3;


### Solution 2


In [ ]:
WITH x AS (
 SELECT e.*, DENSE_RANK() OVER(PARTITION BY department_id ORDER BY salary DESC) rnk
 FROM employees e
) SELECT * FROM x WHERE rnk<=3;


### Solution 3


In [ ]:
SELECT employee_id, employee_name, salary,
       NTILE(4) OVER(ORDER BY salary) salary_quartile
FROM employees;


### Solution 4


In [ ]:
WITH m AS (
 SELECT DATE_FORMAT(o.order_date,'%Y-%m') month,
        SUM(oi.quantity*oi.unit_price) revenue
 FROM orders o JOIN order_items oi ON oi.order_id=o.order_id
 GROUP BY DATE_FORMAT(o.order_date,'%Y-%m')
), x AS (
 SELECT month,revenue,LAG(revenue) OVER(ORDER BY month) prev_revenue FROM m
)
SELECT *, revenue-prev_revenue AS change_amount,
       ROUND(100*(revenue-prev_revenue)/NULLIF(prev_revenue,0),2) pct_change
FROM x;


### Solution 5


In [ ]:
SELECT c.customer_id,
       MIN(o.order_date) AS first_order,
       MAX(o.order_date) AS latest_order
FROM customers c JOIN orders o ON o.customer_id=c.customer_id
GROUP BY c.customer_id;


### Solution 6


In [ ]:
WITH x AS (
 SELECT o.customer_id,o.order_id,o.order_date,
        SUM(oi.quantity*oi.unit_price) order_value
 FROM orders o JOIN order_items oi ON oi.order_id=o.order_id
 GROUP BY o.customer_id,o.order_id,o.order_date
), y AS (
 SELECT x.*, AVG(order_value) OVER(
   PARTITION BY customer_id ORDER BY order_date,order_id
   ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) moving_avg
 FROM x
) SELECT * FROM y;


### Solution 7


In [ ]:
WITH x AS (
 SELECT customer_id, customer_name,
        ROW_NUMBER() OVER(PARTITION BY customer_email ORDER BY customer_id) rn
 FROM customers
) SELECT * FROM x WHERE rn=1;


## Interview Questions & Answers

**Q: WHERE vs HAVING?**  
A: WHERE filters rows before grouping; HAVING filters grouped results.

**Q: RANK vs DENSE_RANK?**  
A: Both assign equal ranks to ties; RANK leaves gaps after ties, DENSE_RANK does not.

**Q: Why can joins create duplicates?**  
A: A one-to-many relationship legitimately creates multiple result rows per parent. First define the intended row grain, then aggregate/deduplicate as required.

**Q: Cursor or set-based SQL?**  
A: Prefer set-based operations for bulk relational transformations; use cursors when row-by-row procedural behavior is genuinely required.

**Q: What does EXPLAIN provide?**  
A: It exposes the optimizer's planned access path, including table access, candidate indexes, selected index, estimated rows and other plan details.


## 5-Day Deliverable Checklist

- [ ] All practice queries executed manually
- [ ] All assignments solved without copying
- [ ] Solutions compared and rewritten
- [ ] At least 10 query variations created
- [ ] At least 5 window-function problems solved
- [ ] At least 3 stored procedures created
- [ ] At least 1 cursor + handler implemented
- [ ] Transactions tested with COMMIT and ROLLBACK
- [ ] EXPLAIN used before/after an index
- [ ] Final capstone query set completed
